# 01 · Data Preparation
**LPS Paper — Zero-Shot Persona Emulation from Conversational Data**

This notebook loads the WildChat-1M dataset, filters for users with sufficient
longitudinal history, and constructs the train/test split used in all experiments.

**Cohort criteria (Section 4.1):**
- ≥ 30 conversations per user
- History spans ≥ 60 days
- 5 most recent conversations held out as test queries Q


In [ ]:

import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
from datasets import load_dataset
from datetime import datetime, timezone
from tqdm import tqdm
import yaml, json, os

# Load config
with open("../configs/default.yaml") as f:
    cfg = yaml.safe_load(f)

print("Config loaded:")
print(f"  min_conversations : {cfg['data']['min_conversations']}")
print(f"  min_days_span     : {cfg['data']['min_days_span']}")
print(f"  holdout_convs     : {cfg['data']['holdout_conversations']}")


In [ ]:

# ── Load WildChat from HuggingFace ────────────────────────────────────
print("Loading WildChat dataset (non-toxic split)...")
dataset = load_dataset("allenai/WildChat", split="train")
print(f"Total conversations: {len(dataset):,}")
df = dataset.to_pandas()
df.head(2)


In [ ]:

# ── Inspect schema ────────────────────────────────────────────────────
print("Columns:", df.columns.tolist())
print("\nSample conversation structure:")
sample = df.iloc[0]
print(f"  conversation_id : {sample['conversation_id']}")
print(f"  timestamp       : {sample['timestamp']}")
print(f"  turns           : {len(sample['conversation'])}")
for turn in sample['conversation'][:2]:
    role = turn['role']
    content = turn['content'][:80]
    print(f"  [{role}] {content}...")


In [ ]:

# ── Extract (user_id, conversation_id, timestamp, turns) ─────────────
records = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
    uid = str(row.get("hashed_ip", row["conversation_id"]))
    ts  = pd.Timestamp(row["timestamp"]).timestamp()
    turns = row["conversation"]
    # Extract all (user_turn, model_turn) pairs
    pairs = []
    for i in range(0, len(turns) - 1, 2):
        if turns[i]["role"] == "user" and turns[i+1]["role"] == "assistant":
            pairs.append({
                "user":  turns[i]["content"],
                "model": turns[i+1]["content"],
            })
    if pairs:
        records.append({
            "user_id": uid,
            "conv_id": row["conversation_id"],
            "timestamp": ts,
            "pairs": pairs,
        })

print(f"Extracted {len(records):,} conversations with interaction pairs")


In [ ]:

# ── Filter cohort: ≥30 conversations, ≥60 days span ──────────────────
from collections import defaultdict

user_convs = defaultdict(list)
for r in records:
    user_convs[r["user_id"]].append(r)

MIN_CONVS = cfg["data"]["min_conversations"]
MIN_DAYS  = cfg["data"]["min_days_span"]
HOLDOUT   = cfg["data"]["holdout_conversations"]

cohort = {}
for uid, convs in user_convs.items():
    convs_sorted = sorted(convs, key=lambda x: x["timestamp"])
    if len(convs_sorted) < MIN_CONVS:
        continue
    span_days = (convs_sorted[-1]["timestamp"] - convs_sorted[0]["timestamp"]) / 86400
    if span_days < MIN_DAYS:
        continue
    cohort[uid] = {
        "history": convs_sorted[:-HOLDOUT],
        "test":    convs_sorted[-HOLDOUT:],
    }

print(f"Cohort size: {len(cohort):,} users")
print(f"Avg history conversations per user: "
      f"{np.mean([len(v['history']) for v in cohort.values()]):.1f}")


In [ ]:

# ── Save cohort ───────────────────────────────────────────────────────
os.makedirs("../data", exist_ok=True)
with open("../data/cohort.json", "w") as f:
    # Save first 100 users for demo (full cohort is 4,200)
    cohort_sample = dict(list(cohort.items())[:100])
    json.dump(cohort_sample, f)

print(f"Saved {len(cohort_sample)} users to ../data/cohort.json")
print("\n✓ Notebook 01 complete — ready for embedding in Notebook 02")
